# 🐼 DuckDB with Pandas & Polars

This notebook demonstrates how to use DuckDB with popular DataFrame libraries.

## Why DuckDB + DataFrames?

- **SQL power**: Write complex analytical queries with SQL
- **Zero-copy**: DuckDB can query DataFrames without copying data
- **Speed**: DuckDB is often faster than pandas for analytical queries
- **Flexibility**: Switch between SQL and DataFrame APIs seamlessly

In [ ]:
import duckdb
import pandas as pd

# Try to import polars (optional)
try:
    import polars as pl
    POLARS_AVAILABLE = True
    print(f"Polars version: {pl.__version__}")
except ImportError:
    POLARS_AVAILABLE = False
    print("Polars not installed. Install with: pip install polars")

print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Pandas Integration

### Query Pandas DataFrames with SQL

In [ ]:
# Create sample DataFrames
orders_df = pd.DataFrame({
    'order_id': range(1, 1001),
    'customer_id': [i % 100 + 1 for i in range(1000)],
    'product_id': [i % 50 + 1 for i in range(1000)],
    'quantity': [1 + (i % 10) for i in range(1000)],
    'unit_price': [10.0 + (i % 20) * 5 for i in range(1000)],
    'order_date': pd.date_range('2024-01-01', periods=1000, freq='h')
})

customers_df = pd.DataFrame({
    'customer_id': range(1, 101),
    'name': [f'Customer {i}' for i in range(1, 101)],
    'city': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'] * 20,
    'segment': ['Consumer', 'Corporate', 'Home Office'] * 33 + ['Consumer']
})

print(f"Orders: {len(orders_df)} rows")
print(f"Customers: {len(customers_df)} rows")
orders_df.head()

In [ ]:
# Query DataFrame directly with SQL - DuckDB finds variables automatically!
result = duckdb.query("""
    SELECT 
        c.city,
        c.segment,
        COUNT(DISTINCT o.order_id) as order_count,
        SUM(o.quantity * o.unit_price) as total_revenue,
        AVG(o.quantity * o.unit_price) as avg_order_value
    FROM orders_df o
    JOIN customers_df c ON o.customer_id = c.customer_id
    GROUP BY c.city, c.segment
    ORDER BY total_revenue DESC
""").df()

print("Sales by City and Segment:")
result

### Window Functions on DataFrames

Window functions are much easier in SQL than in pandas!

In [ ]:
# Calculate running totals and rankings
result = duckdb.query("""
    SELECT 
        order_date::DATE as date,
        SUM(quantity * unit_price) as daily_revenue,
        SUM(SUM(quantity * unit_price)) OVER (ORDER BY order_date::DATE) as running_total,
        ROW_NUMBER() OVER (ORDER BY SUM(quantity * unit_price) DESC) as revenue_rank
    FROM orders_df
    GROUP BY order_date::DATE
    ORDER BY date
    LIMIT 10
""").df()

print("Daily Revenue with Running Total:")
result

### Convert Between Pandas and DuckDB

In [ ]:
# Create connection and register DataFrame
conn = duckdb.connect()

# Register DataFrame as a virtual table
conn.register('orders', orders_df)
conn.register('customers', customers_df)

# Now query using the registered names
result = conn.execute("SELECT COUNT(*) as total FROM orders").fetchone()
print(f"Registered table has {result[0]} rows")

# Create a DuckDB table from DataFrame
conn.execute("CREATE TABLE orders_table AS SELECT * FROM orders")
print("Created persistent table from DataFrame")

## 2. Polars Integration

DuckDB also works seamlessly with Polars DataFrames!

In [ ]:
if POLARS_AVAILABLE:
    # Create Polars DataFrame
    orders_pl = pl.DataFrame({
        'order_id': range(1, 1001),
        'customer_id': [i % 100 + 1 for i in range(1000)],
        'amount': [100.0 + (i % 50) * 10 for i in range(1000)]
    })
    
    # Query Polars DataFrame with DuckDB
    result = duckdb.query("""
        SELECT 
            customer_id,
            COUNT(*) as order_count,
            SUM(amount) as total_amount
        FROM orders_pl
        GROUP BY customer_id
        ORDER BY total_amount DESC
        LIMIT 5
    """).pl()  # Return as Polars DataFrame!
    
    print("Top 5 Customers (Polars):")
    print(result)
else:
    print("Polars not available - skipping this section")

## 3. Performance Comparison

Let's compare DuckDB vs pure pandas for an analytical query.

In [ ]:
import time

# Create larger dataset for benchmarking
n_rows = 100_000
large_df = pd.DataFrame({
    'category': [f'cat_{i % 100}' for i in range(n_rows)],
    'value': [float(i % 1000) for i in range(n_rows)],
    'date': pd.date_range('2020-01-01', periods=n_rows, freq='min')
})

print(f"Dataset size: {len(large_df):,} rows")

In [ ]:
# Pandas approach
start = time.time()
pandas_result = (large_df
    .groupby('category')
    .agg({'value': ['sum', 'mean', 'count']})
    .reset_index()
)
pandas_time = time.time() - start

# DuckDB approach
start = time.time()
duckdb_result = duckdb.query("""
    SELECT 
        category,
        SUM(value) as value_sum,
        AVG(value) as value_mean,
        COUNT(*) as value_count
    FROM large_df
    GROUP BY category
""").df()
duckdb_time = time.time() - start

print(f"Pandas time:  {pandas_time:.4f}s")
print(f"DuckDB time:  {duckdb_time:.4f}s")
print(f"DuckDB is {pandas_time/duckdb_time:.1f}x faster")

## 4. Real-World Example: AdventureWorks Analysis

Let's analyze the AdventureWorks dataset combining multiple DataFrames.

In [ ]:
# Load AdventureWorks data as pandas DataFrames
products = pd.read_csv('../sample_data/AW_CSV/Production.Product.csv')
categories = pd.read_csv('../sample_data/AW_CSV/Production.ProductCategory.csv')
subcategories = pd.read_csv('../sample_data/AW_CSV/Production.ProductSubcategory.csv')
order_details = pd.read_csv('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv')

print(f"Products: {len(products)} rows")
print(f"Categories: {len(categories)} rows")
print(f"Subcategories: {len(subcategories)} rows")
print(f"Order Details: {len(order_details)} rows")

In [ ]:
# Complex multi-table analysis with SQL
result = duckdb.query("""
    WITH product_sales AS (
        SELECT 
            p.ProductID,
            p.Name as ProductName,
            p.ListPrice,
            sc.Name as Subcategory,
            c.Name as Category,
            COALESCE(SUM(od.OrderQty), 0) as TotalQtySold,
            COALESCE(SUM(od.LineTotal), 0) as TotalRevenue
        FROM products p
        LEFT JOIN subcategories sc ON p.ProductSubcategoryID = sc.ProductSubcategoryID
        LEFT JOIN categories c ON sc.ProductCategoryID = c.ProductCategoryID
        LEFT JOIN order_details od ON p.ProductID = od.ProductID
        GROUP BY p.ProductID, p.Name, p.ListPrice, sc.Name, c.Name
    )
    SELECT 
        Category,
        Subcategory,
        COUNT(*) as ProductCount,
        SUM(TotalQtySold) as TotalQuantity,
        ROUND(SUM(TotalRevenue), 2) as TotalRevenue,
        ROUND(AVG(ListPrice), 2) as AvgListPrice
    FROM product_sales
    WHERE Category IS NOT NULL
    GROUP BY Category, Subcategory
    ORDER BY TotalRevenue DESC
    LIMIT 10
""").df()

print("Top 10 Subcategories by Revenue:")
result

## 🎯 Summary

| Feature | Pandas | DuckDB on DataFrame |
|---------|--------|---------------------|
| JOINs | Complex syntax | Simple SQL |
| Window Functions | Difficult | Native support |
| Aggregations | Method chaining | SQL GROUP BY |
| Performance | Single-threaded | Multi-threaded |
| Subqueries | Not native | Full support |
| CTEs | Not available | Full support |

### When to Use DuckDB with DataFrames

✅ **Use DuckDB when:**
- Complex JOINs across multiple tables
- Window functions (ranking, running totals)
- Large aggregations
- SQL is more readable than pandas

✅ **Use pandas when:**
- Simple row-by-row transformations
- String manipulation
- Visualization integration
- Index-based operations

### Best of Both Worlds

```python
# Start with pandas for data loading
df = pd.read_csv('data.csv')

# Use DuckDB for complex analysis
result = duckdb.query("SELECT ... FROM df ...").df()

# Back to pandas for visualization
result.plot()
```

## 🦆 Happy Quacking!